# V2 Benchmark出题：候选设计与原图检索

最终知识 → 一次候选设计 →（仅编辑：搜原图、核验、定稿）→ 校验与审核 → 作答检索 → 导出。

**输入不再包含手写plan、intent、选材ID或预先匹配的目标图。**全局seed、预算、题型、模型及声明的原图检索源属于运行配置。任务由知识、条件与视觉推理支撑；简单直接的知识应用也可成立，明确执行要求只作辅助，以新题实际检查信号、题面和原判据的对应，不要求training-free提升。未发布知识不会进入模型；配图、编辑原图、监督目标分别管理。已有知识本身仍有质量限制，因此新产物称开发候选。

从下面第一个代码cell开始；默认只读实际保存结果。所有展示使用Markdown、表格和原生图片，不使用HTML。

本册每一步直接执行与整体版相同的算子；按输入、prompt、调用、输出排列。`plan`仅为选题后派生的类型／划分元数据，不能在入口手填。



**当前按指导信号与训练候选思路迭代。** 新题用于检查材料、任务、公开展示条件与原题判据的对应；失败可用于研究训练，但不等于正确监督已备齐。作答检索与评测共享公开题面检索实现。旧案例仍按冻结版本解释；新版执行使用新的NEW_RUN。

当前主要prompt：[候选设计](prompts/design_candidates.md)、[原图审核（本地与外搜共用）](prompts/select_edit_source.md)、[编辑定稿](prompts/construct.md)、[任务审核](prompts/review_task.md)。逐步版每个模型算子前展示当前prompt全文；旧案例的实际输入以其冻结请求为准。

In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), *Path.cwd().parents, Path("/yzp/zhaozy/yangzepeng/0905/demiwtg")]
PROJECT = next((p for p in _candidates if (p / "curation/benchmark/authoring.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("找不到包含curation的项目根目录")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
try:
    from curation.preparation.inspection import show_records, show_prompt, show_summary, snapshot_ref, check_step_order
    from curation.benchmark.runtime import config as build_config
    from curation.preparation.records import run_records, run_manifest, rows
except ModuleNotFoundError as error:
    raise RuntimeError("请选择demiwtg内核：/yzp/zhaozy/yangzepeng/0905/env/bin/python") from error

MODE = "view_saved"  # execute运行；view_saved只读已有checkpoint。
BASE = PROJECT / "curation/benchmark/runs"
SAVED_RUN = BASE / "pipeline_v2_benchmark_review"
NEW_RUN = PROJECT / "curation/benchmark/runs/pipeline_v2_benchmark_review"  # 输入／代码／prompt变化须新run。
knowledge_runs = []  # 填入知识发布 ID 或固定 DatasetRef；视觉发布可另传 visual_runs。
# 知识输入为最终交付；编辑原图检索的全库清单及外部源在下面config中公开配置。
# 发现和选题只读最终知识；原始场景／目标图片仅在后续显式素材步骤按需读取。
MODEL_BACKEND = "offline"  # local用本地Qwen；offline为相同prompt／context生成绑定请求。
# concepts=None使用全部发布概念；可传名称列表做可复现小批。
config = build_config(MODEL_BACKEND, concepts=None, seed=0, max_units=2, tasks_per_unit=1, max_context_chars=100000,
    scene_search={"image_ref": None,
                  "external_providers": ["commons"]},
    author_model="gpt-6-astra" if MODEL_BACKEND == "offline" else None,
    author_effort="high" if MODEL_BACKEND == "offline" else None)
CASE_ID = None  # 默认展示本阶段第一条实际参与处理的记录；也可填unit_id或task_id。
SHOW_AUDIT = False  # 完整来源、字段、实际模型请求；图片按角色原样展示。
THROUGH = "export"
if MODE not in {"view_saved", "execute"}:
    raise ValueError("MODE必须是view_saved或execute")
run = SAVED_RUN if MODE == "view_saved" else NEW_RUN
if MODE == "view_saved":
    saved = run_records(run).get("manifest")
    if saved is not None:
        config = saved["config"]
    else:
        print("请指定已有 V2 Lance run；后续查看单元需要已完成的阶段。")
print("项目：", PROJECT, "\n内核：", sys.executable, "\n模式：", MODE, "\n运行：", run)


print("本次运行配置：", config)


## 初始化运行

冻结知识文件、引用来源文件、配置、代码与prompt版本；检查原始像素是否被修改。正式测试保留表若存在可通过config传入；省略时只允许开发用途。

In [ ]:
from functools import partial
from demiflow.standalone import local_data
from curation.preparation.records import run_lock
from curation.benchmark.runtime import graph_version
from curation.benchmark.authoring import AuthoringRunFiles, input_records, split_guard, knowledge_items
from curation.benchmark.candidates import ReadKnowledge, ExpandCandidates
from curation.benchmark.scene_search import SceneSearch, SearchLocalScenes, SearchExternalScenes
from curation.benchmark.operators import ValidateTask, RetrieveForAnswer, export_record
from curation.benchmark.prompting import prompt_config, prompt_responses, prepare_design, apply_design, prepare_external_edit_source, apply_external_edit_source, prepare_edit_source, apply_edit_source, prepare_construct, apply_construct, prepare_task_review, apply_task_review



In [ ]:
if MODE == "execute":
    with run_lock(run):
        files = AuthoringRunFiles(run, knowledge_runs, "benchmark", config, graph_version("benchmark"))
        guard = split_guard(files)
        pack, options = prompt_config(run, config)
        data = local_data(prompt_packs={"tasks.yaml": pack}, prompt_options=options,
                          max_prompt_requests=config["model"]["max_calls"])
else:
    data = local_data()
    saved_manifest = run_manifest(run)
    print("当前展示的冻结输入：", saved_manifest["knowledge_runs"])


## 1. 读取最终知识

| 项目 | 说明 |
|---|---|
| 实际算子 | `map(ReadKnowledge)` |
| 输入与输出 | 已发布的知识发布 DatasetRef → 每概念一份完整知识包 |
| 为什么做／边界 | 正文、最终配图与引用一起读入；不建立重复共享表、不拆发现窗口、不读取原始图片池。无知识和隔离缺口在本checkpoint保留。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
print("知识发布引用：", knowledge_runs)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, None)
        knowledge = (data.from_iter(lambda: input_records(files))
            .map(ReadKnowledge(files.knowledge_version, "benchmark", config, guard))
)
        knowledge = files.lance_checkpoint(knowledge, "knowledge")
        state = files.finish()
else:
    knowledge = data.from_iter(lambda: rows(snapshot_ref(run, "knowledge")))


In [ ]:
show_records(knowledge, "knowledge", CASE_ID, audit=SHOW_AUDIT)


## 2. 一次设计候选任务

| 项目 | 说明 |
|---|---|
| 实际算子 | `filter（公开seed预算）→ map_prompt_async("design_candidates")` |
| 输入与输出 | 完整知识包＋全局预算 → 候选考点与任务设计 |
| 为什么做／边界 | 模型自主选知识空缺、应用方式和可观察判据。T2I直接给题面；edit只给设计、原图需求及查询词。seed和max_units只控制探索范围，无手工计划。超出上下文预算明确停下，不静默截断。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(knowledge, "knowledge", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("design_candidates", branch="benchmark")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'knowledge')
        scope = knowledge.filter(lambda r: r["status"] == "knowledge_available")
        # Author scope must not shrink the shared answer-retrieval catalog.
        if config.get("concepts"):
            scope = scope.filter(lambda r: r["concept"] in config["concepts"])
        if config["max_units"] is not None:
            ranks = scope.map(lambda r: (r["sampling_key"], r["unit_id"])).take_all()
            selected_ids = {unit for _, unit in sorted(ranks)[:config["max_units"]]}
            scope = scope.filter(lambda r: r["unit_id"] in selected_ids)
        designed = (scope
            .map(partial(prepare_design, run=run, pack=pack))
            .map_prompt_async("design_candidates", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="design_candidates_result", call_output="design_candidates_call", error_output="design_candidates_error",
                when=lambda r: r["status"] == "knowledge_available", concurrency=1, queue_depth=1)
            .map(partial(apply_design, run=run))
)
        designed = files.lance_checkpoint(designed, "design", extra=prompt_responses(run, "design_candidates"))
        state = files.finish()
else:
    designed = data.from_iter(lambda: rows(snapshot_ref(run, "design")))


In [ ]:
show_records(designed, "design", CASE_ID, audit=SHOW_AUDIT)


## 3. 展开候选与绑定证据

| 项目 | 说明 |
|---|---|
| 实际算子 | `flat_map(ExpandCandidates)` |
| 输入与输出 | 模型候选列表 → 每题一行 |
| 为什么做／边界 | 这一步不调用模型。校验引用、保留必需配图并重编号；派生类型、划分和ID。T2I已有草稿，edit等待原图，坏候选保留原因。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(designed, "design", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'design')
        candidates = (designed.flat_map(ExpandCandidates(guard, config))
)
        candidates = files.lance_checkpoint(candidates, "candidates")
        state = files.finish()
else:
    candidates = data.from_iter(lambda: rows(snapshot_ref(run, "candidates")))


In [ ]:
show_records(candidates, "candidates", CASE_ID, audit=SHOW_AUDIT)


## 4. 编辑：按初始场景搜本地图

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_async(SearchLocalScenes)` |
| 输入与输出 | 编辑原图需求及查询词＋全库图片清单 → 候选原图 |
| 为什么做／边界 | 跨概念扫描caption／标题／已有概念元数据，排序后只读取有界候选像素。不以当前知识概念限制来源；例如添加对象可找尚无该对象的承载场景。排除参考近重复及测试保留，不把候选当合格原图。T2I跳过。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(candidates, "candidates", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'candidates')
        scene_search = SceneSearch(run, knowledge.flat_map(knowledge_items).take_all(), guard, config)
        edit_candidates = (candidates.map_async(SearchLocalScenes(scene_search), concurrency=1)
)
        edit_candidates = files.lance_checkpoint(edit_candidates, "edit_search")
        state = files.finish()
else:
    edit_candidates = data.from_iter(lambda: rows(snapshot_ref(run, "edit_search")))


In [ ]:
show_records(edit_candidates, "edit_search", CASE_ID, audit=SHOW_AUDIT)


## 5. 仅编辑：核验原图

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_prompt_async("select_edit_source")` |
| 输入与输出 | 知识考点、最终知识、候选原图像素与来源 → 合格原图或needs_edit_source |
| 为什么做／边界 | 这个额外输入只属于编辑场景，不是知识。实际核验来源、非生成性质、可见锚点、可实施的一次主操作；不能根据图片回头编造知识。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(edit_candidates, "edit_search", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("select_edit_source", branch="benchmark")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'edit_search')
        source_ready = (edit_candidates
            .map(partial(prepare_edit_source, run=run, pack=pack))
            .map_prompt_async("select_edit_source", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="select_edit_source_result", call_output="select_edit_source_call", error_output="select_edit_source_error",
                when=lambda r: r["status"] == "edit_candidates_ready", concurrency=1, queue_depth=1)
            .map(partial(apply_edit_source, run=run))
)
        source_ready = files.lance_checkpoint(source_ready, "edit_source", extra=prompt_responses(run, "select_edit_source"))
        state = files.finish()
else:
    source_ready = data.from_iter(lambda: rows(snapshot_ref(run, "edit_source")))


In [ ]:
show_records(source_ready, "edit_source", CASE_ID, audit=SHOW_AUDIT)


## 6. 编辑：必要时外部补搜

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_async(SearchExternalScenes)` |
| 输入与输出 | 本地无候选或像素审核拒绝 → 外部候选或可追溯缺口 |
| 为什么做／边界 | 默认Wikimedia Commons，可配置SearxNG图片搜索。使用模型已给出的初始场景查询；搜索数、下载数与字节数有界。来源、许可、实际像素、错误均冻结；续跑不重复请求。不是生成底图。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(source_ready, "edit_source", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'edit_source')
        external_candidates = (source_ready.map_async(SearchExternalScenes(scene_search), concurrency=1)
)
        external_candidates = files.lance_checkpoint(external_candidates, "edit_external_search")
        state = files.finish()
else:
    external_candidates = data.from_iter(lambda: rows(snapshot_ref(run, "edit_external_search")))


In [ ]:
show_records(external_candidates, "edit_external_search", CASE_ID, audit=SHOW_AUDIT)


## 7. 编辑：核验外部原图

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_prompt_async("select_edit_source_external")` |
| 输入与输出 | 外部候选像素及来源 → 可用原图或缺口 |
| 为什么做／边界 | 与本地原图共用同一业务prompt和准入要求；独立命名只区分不同请求输入与断点。看不清锚点、来源不足或任务不可实施就保留缺口。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(external_candidates, "edit_external_search", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("select_edit_source_external", branch="benchmark")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'edit_external_search')
        all_sources = (external_candidates
            .map(partial(prepare_external_edit_source, run=run, pack=pack))
            .map_prompt_async("select_edit_source_external", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="select_edit_source_external_result", call_output="select_edit_source_external_call", error_output="select_edit_source_external_error",
                when=lambda r: r["status"] == "external_edit_candidates_ready", concurrency=1, queue_depth=1)
            .map(partial(apply_external_edit_source, run=run))
)
        all_sources = files.lance_checkpoint(all_sources, "edit_external_source", extra=prompt_responses(run, "select_edit_source_external"))
        state = files.finish()
else:
    all_sources = data.from_iter(lambda: rows(snapshot_ref(run, "edit_external_source")))


In [ ]:
show_records(all_sources, "edit_external_source", CASE_ID, audit=SHOW_AUDIT)


## 8. 编辑：据真实原图定稿

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_prompt_async("construct")` |
| 输入与输出 | 候选设计＋选定原图像素 → 具体题面和判据 |
| 为什么做／边界 | 只在编辑原图审核通过后执行，绑定实际锚点、一次主操作和必要保持。T2I沿用design直接给出的草稿，不再重复构题。不能为迁就原图改知识或看目标倒写题。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(all_sources, "edit_external_source", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("construct", branch="benchmark")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'edit_external_source')
        constructed = (all_sources
            .map(partial(prepare_construct, run=run, pack=pack))
            .map_prompt_async("construct", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="construct_result", call_output="construct_call", error_output="construct_error",
                when=lambda r: r["status"] == "selected", concurrency=1, queue_depth=1)
            .map(partial(apply_construct, run=run))
)
        constructed = files.lance_checkpoint(constructed, "construct", extra=prompt_responses(run, "construct"))
        state = files.finish()
else:
    constructed = data.from_iter(lambda: rows(snapshot_ref(run, "construct")))


In [ ]:
show_records(constructed, "construct", CASE_ID, audit=SHOW_AUDIT)


## 9. 校验任务契约

| 项目 | 说明 |
|---|---|
| 实际算子 | `map(ValidateTask)` |
| 输入与输出 | 草稿和材料编号 → 绑定稳定知识ID的判据及冻结任务哈希 |
| 为什么做／边界 | 检查字段、来源编号、编辑类型、隔离与禁止的旧配额字段。程序通过只说明契约合法，不证明知识语义正确。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(constructed, "construct", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'construct')
        validated = (constructed.map(ValidateTask(guard))
)
        validated = files.lance_checkpoint(validated, "validate")
        state = files.finish()
else:
    validated = data.from_iter(lambda: rows(snapshot_ref(run, "validate")))


In [ ]:
show_records(validated, "validate", CASE_ID, audit=SHOW_AUDIT)


## 10. 审核任务质量

| 项目 | 说明 |
|---|---|
| 实际算子 | `map_prompt_async("review_task")` |
| 输入与输出 | 草稿、全部构题证据、编辑原图 → 五项语义审核 |
| 为什么做／边界 | 检查来源支持、知识必要、可观察、不泄漏、保持合理。新请求无作者聊天；通过仍是reviewed_candidate_not_golden，不授予人工golden。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(validated, "validate", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
show_prompt("review_task", branch="benchmark")


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'validate')
        reviewed = (validated
            .map(partial(prepare_task_review, run=run, pack=pack))
            .map_prompt_async("review_task", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="review_task_result", call_output="review_task_call", error_output="review_task_error",
                when=lambda r: r["status"] == "valid_task", concurrency=1, queue_depth=1)
            .map(partial(apply_task_review, run=run))
)
        reviewed = files.lance_checkpoint(reviewed, "review", extra=prompt_responses(run, "review_task"))
        state = files.finish()
else:
    reviewed = data.from_iter(lambda: rows(snapshot_ref(run, "review")))


In [ ]:
show_records(reviewed, "review", CASE_ID, audit=SHOW_AUDIT)


## 11. 按题面检索作答参考

| 项目 | 说明 |
|---|---|
| 实际算子 | `map(RetrieveForAnswer)` |
| 输入与输出 | 公开instruction与共享知识表 → 实际作答材料及检索差距 |
| 为什么做／边界 | 只有公开题面参与词项排序；隐藏考点、判据、答案和目标caption不参与。检索后另记是否覆盖构题证据，不能将手选材料当检索结果。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(reviewed, "review", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'review')
        retrieved = (reviewed.map(RetrieveForAnswer(knowledge.flat_map(knowledge_items).take_all(), guard))
)
        retrieved = files.lance_checkpoint(retrieved, "retrieve")
        state = files.finish()
else:
    retrieved = data.from_iter(lambda: rows(snapshot_ref(run, "retrieve")))


In [ ]:
show_records(retrieved, "retrieve", CASE_ID, audit=SHOW_AUDIT)


## 12. 导出并保留缺口

| 项目 | 说明 |
|---|---|
| 实际算子 | `map(export_record) → filter` |
| 输入与输出 | 任务及各阶段审核 → ready／incomplete |
| 为什么做／边界 | Benchmark输出题面和判据；训练输出知识／原图输入与仅在目标计算loss的序列。未采用、失败、待响应和缺目标保留。逐项K评分与BAGEL读取器不属于这次修改。 |

以下先查看本步输入，再执行算子，最后查看结果。

In [ ]:
show_records(retrieved, "retrieve", CASE_ID, audit=SHOW_AUDIT)


In [ ]:
if MODE == "execute":
    with run_lock(run):
        check_step_order(files, 'retrieve')
        exported = (retrieved.map(export_record)
)
        exported = files.lance_checkpoint(exported, "export")
        ready = (exported.filter(lambda r: r["export_ready"])
)
        ready = files.lance_checkpoint(ready, "ready")
        incomplete = (exported.filter(lambda r: not r["export_ready"])
)
        incomplete = files.lance_checkpoint(incomplete, "incomplete")
        state = files.finish()
else:
    exported = data.from_iter(lambda: rows(snapshot_ref(run, "export")))


In [ ]:
show_records(exported, "export", CASE_ID, audit=SHOW_AUDIT)


## 全链路概览

续跑时从初始化开始，依次执行；已有checkpoint自动复用。新增offline响应仅使对应模型步及其下游产生新revision，旧结果原样保留。

In [ ]:
show_summary(run)
